# Frozen specifications and the one-time final test (Prompt #10, Steps 71-80, 91-94, 100)

The freeze rules were written in `config/ml.yaml` before any result: at least four of five blocks beating the constant, then the simplest model within one SE of the best. Each frozen spec (MODEL_SPEC_*.json, content-hashed, immutable) is fitted on the whole development span, saved in its native format, reloaded, replayed bar by bar (streaming = batch) and timed - and only then evaluated once on the reserved period. Every access to the reserved period is in `final_test/access_log.jsonl`.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.ml` and `xauusd_quant.research.ml_*` and read back from `results/ml_research/`. Generate those first:

```
xq ml-research --timeframe 5m [--stages grid,trees,...]   # cached walk-forward units
xq ml-report --timeframe 5m                              # tables, ML-H ledger rows, plots
xq ml-compare --timeframe 5m --target mean_reversion      # one target, from the units
xq ml-freeze --timeframe 5m                              # pre-registered freeze rules
xq ml-finalize --timeframe 5m                            # artifacts, streaming, latency
xq ml-final-test --model-spec LGBM_REVERSION_5M_H5_V001   # once, frozen specs only
```

**Prediction research only.** Models output probabilities and expected values; there is no BUY / SELL, threshold, position, stop, PnL or execution anywhere. Walk-forward blocks 2011-2021 are development; the reserved period (2022-) is read only by `ml-final-test`, after the specs are frozen. Skill is measured against the constant training mean (log-loss / MSE skill, positive = better than the baseline) and read block by block.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.ml.config import load_ml_config

cfg = load_ml_config(ROOT / "config" / "ml.yaml")

TIMEFRAME = "5m"       # 5m | 15m - change and re-run

out = cfg.results_path / TIMEFRAME
if not (out / "tables").exists():
    print(f"No report for {TIMEFRAME}. Run:  xq ml-research --timeframe {TIMEFRAME} --report")

def table(name):
    """Load a previously generated table (None if it was not written)."""
    path = out / "tables" / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(*parts):
    path = out.joinpath(*parts)
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Freeze decisions

In [ ]:
freeze = load_json('frozen', 'freeze_report.json')
pl.DataFrame([{k: d.get(k) for k in ('target', 'horizon', 'eligible', 'configurations', 'frozen',
              'reason', 'window_rule', 'hyperparameter_rule', 'calibration_rule')}
              for d in freeze.get('decisions', [])], infer_schema_length=None)

## Final development models: reload, streaming, latency

In [ ]:
fin = load_json('frozen', 'finalize.json')
pl.DataFrame([{k: m.get(k) for k in ('model_id', 'role', 'reload_equal', 'streaming_safe',
              'model_size_bytes', 'latency_ms_per_row', 'latency_ms_per_10k_rows')}
              for m in fin.get('models', [])], infer_schema_length=None)

## Model summaries: development vs final test (Step 100)

In [ ]:
pl.DataFrame(list(load_json('final_test', 'model_summaries.json').values()), infer_schema_length=None)

## Final test by year and the most recent twelve months

In [ ]:
for p in sorted((out / 'final_test').glob('*_V0*.json')):
    r = json.loads(p.read_text())
    print(r['model_id'], {y: round(m.get('log_loss_skill') or m.get('rank_ic') or 0, 4)
                          for y, m in r['by_year'].items()})

## Access log

In [ ]:
print((out / 'final_test' / 'access_log.jsonl').read_text() if (out / 'final_test' / 'access_log.jsonl').exists() else '(final test not run)')